In [ ]:
"""
train.py
--------
Trains all three models (LogReg, XGBoost, MLP) and saves results.

Usage:
    python src/train.py --model all --city sh
    python src/train.py --model mlp --city sh --epochs 30

Design notes:
- Time-based train/val/test split (no random shuffle) to simulate production:
  train on early months, evaluate on later months.
- pos_weight is computed from training data to handle class imbalance.
- All artifacts (model, scaler, encoders) are saved for inference.
"""

In [ ]:
import argparse
import json
import pickle
from pathlib import Path

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from xgboost import XGBClassifier
from tqdm import tqdm

In [ ]:
from data_loader import get_data
from features import build_features, FEATURE_COLS
from model import build_model
from evaluate import compute_metrics, find_best_threshold

In [ ]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)

---------------------------------------------------------------------------
Train / val / test split (time-based)
---------------------------------------------------------------------------

In [ ]:
def time_split(df, train_frac=0.7, val_frac=0.15):
    """
    Split by time rather than randomly.
    This prevents leakage and simulates real deployment conditions.
    """
    df = df.sort_values("accept_time").reset_index(drop=True)
    n = len(df)
    i_train = int(n * train_frac)
    i_val   = int(n * (train_frac + val_frac))
    return df.iloc[:i_train], df.iloc[i_train:i_val], df.iloc[i_val:]

---------------------------------------------------------------------------
PyTorch training loop
---------------------------------------------------------------------------

In [ ]:
def train_mlp(X_train, y_train, X_val, y_val,
              epochs: int = 20, batch_size: int = 2048, lr: float = 1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training MLP on {device} …")

    # Class imbalance: how many negatives per positive?
    pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    print(f"  pos_weight = {pos_weight:.2f}")

    model, loss_fn = build_model(
        input_dim=X_train.shape[1],
        pos_weight=float(pos_weight),
        use_focal=True,
    )
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(X_train) // batch_size + 1, epochs=epochs
    )

    # Datasets & loaders
    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 4)

    best_auc_pr = 0
    best_state  = None
    patience    = 5
    no_improve  = 0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        # Validation
        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                probs = model.predict_proba(xb.to(device)).cpu().numpy()
                val_probs.extend(probs)
                val_labels.extend(yb.numpy())

        auc_pr = average_precision_score(val_labels, val_probs)
        print(f"  Epoch {epoch:02d} | loss={epoch_loss/len(train_loader):.4f} | val AUC-PR={auc_pr:.4f}")

        if auc_pr > best_auc_pr:
            best_auc_pr = auc_pr
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve  = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model

---------------------------------------------------------------------------
Baseline models
---------------------------------------------------------------------------

In [ ]:
def train_logreg(X_train, y_train):
    print("Training Logistic Regression …")
    clf = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        C=0.1,
    )
    clf.fit(X_train, y_train)
    return clf

In [ ]:
def train_xgboost(X_train, y_train):
    print("Training XGBoost …")
    scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    clf = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=float(scale_pos),
        eval_metric="aucpr",
        early_stopping_rounds=20,
        verbosity=0,
    )
    clf.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)
    return clf

---------------------------------------------------------------------------
Main
---------------------------------------------------------------------------

In [ ]:
def run(city: str = "sh", models: list[str] | None = None, epochs: int = 20):
    if models is None:
        models = ["logreg", "xgboost", "mlp"]

    print(f"\n=== LaDe Late-Delivery Classifier | city={city} ===\n")
    df = get_data(city=city)

    train_df, val_df, test_df = time_split(df)
    print(f"Split: train={len(train_df):,} | val={len(val_df):,} | test={len(test_df):,}")

    X_train, y_train, encoders, scaler = build_features(train_df, fit=True)
    X_val,   y_val,   _,        _      = build_features(val_df,   encoders=encoders, scaler=scaler, fit=False)
    X_test,  y_test,  _,        _      = build_features(test_df,  encoders=encoders, scaler=scaler, fit=False)

    # Save preprocessing artifacts for inference
    with open(MODELS_DIR / "encoders.pkl", "wb") as f:
        pickle.dump(encoders, f)
    with open(MODELS_DIR / "scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)

    all_metrics = {}

    if "logreg" in models:
        clf = train_logreg(X_train, y_train)
        probs = clf.predict_proba(X_test)[:, 1]
        thresh = find_best_threshold(y_val, clf.predict_proba(X_val)[:, 1])
        metrics = compute_metrics(y_test, probs, threshold=thresh, name="LogReg")
        all_metrics["logreg"] = metrics
        with open(MODELS_DIR / "logreg.pkl", "wb") as f:
            pickle.dump(clf, f)

    if "xgboost" in models:
        clf = train_xgboost(X_train, y_train)
        probs = clf.predict_proba(X_test)[:, 1]
        thresh = find_best_threshold(y_val, clf.predict_proba(X_val)[:, 1])
        metrics = compute_metrics(y_test, probs, threshold=thresh, name="XGBoost")
        all_metrics["xgboost"] = metrics
        clf.save_model(str(MODELS_DIR / "xgboost.json"))

    if "mlp" in models:
        mlp = train_mlp(X_train, y_train, X_val, y_val, epochs=epochs)
        torch.save(mlp.state_dict(), MODELS_DIR / "mlp.pt")
        mlp.eval()
        device = next(mlp.parameters()).device
        with torch.no_grad():
            val_probs  = mlp.predict_proba(torch.tensor(X_val,  dtype=torch.float32).to(device)).cpu().numpy()
            test_probs = mlp.predict_proba(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()
        thresh = find_best_threshold(y_val, val_probs)
        metrics = compute_metrics(y_test, test_probs, threshold=thresh, name="MLP")
        all_metrics["mlp"] = metrics

    print("\n=== Final Test Results ===")
    for name, m in all_metrics.items():
        print(f"  {name:10s} | AUC-PR={m['auc_pr']:.4f} | AUC-ROC={m['auc_roc']:.4f} | F1={m['f1']:.4f}")

    with open(OUTPUTS_DIR / "metrics.json", "w") as f:
        json.dump(all_metrics, f, indent=2)
    print(f"\nMetrics saved to {OUTPUTS_DIR / 'metrics.json'}")

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--city",   default="sh")
    parser.add_argument("--model",  default="all",
                        help="logreg | xgboost | mlp | all")
    parser.add_argument("--epochs", type=int, default=20)
    args = parser.parse_args()
    models = ["logreg", "xgboost", "mlp"] if args.model == "all" else [args.model]
    run(city=args.city, models=models, epochs=args.epochs)